In [ ]:
import sys
import os

# 1. Check if colab is already working
if 'google.colab' in str(get_ipython()):
    #Import both the url of the repository and its name
    
    REPO_URL = "https://github.com/aledelma99/DQN-Project"
    REPO_NAME = "DQN-Project"
    
    
    # Clone the repository inside colab if not present
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}
        
    # Set up the save link for the checkpoint inside Github
    project_path = f"/content/{REPO_NAME}"
    checkpoint_path = os.path.join(project_path, "checkpoints")
    
    
    #Load the Github repository
    os.chdir(project_path)
    print("Repository cloned correctly!")
else:
    
    project_path = os.getcwd()
    checkpoint_path = os.path.join(project_path, "checkpoints")

#Check if the checkpoint folder exists
os.makedirs(checkpoint_path, exist_ok=True)


# Update the path in order to retrieve the file .py
if project_path not in sys.path:
    sys.path.append(project_path)

#Import the functions used in this notebook
from DQN_tools import Network, ReplayMemory, Agent, compute_avg_q, save_model, train_agent

print("Workplace correctly loaded")
print(f" THe training weights will be taken by: {checkpoint_path}")

In [ ]:
!pip install swig
!pip install gymnasium[box2d]
import importlib
import subprocess
import sys

required_packages = [
    "gymnasium",
    "numpy",
    "torch",
    "matplotlib",
]

for package in required_packages:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f" The library '{package}' was not found, install it")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f" '{package}' is installed")

print("\n Import all the necessary libraries")

import os
import random
import time
from collections import deque
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.autograd as autograd
from torch.autograd import Variable
import matplotlib.pyplot as plt
from google.colab import drive
import zipfile


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 89.6 MB/s eta 0:00:00

 Import all the necessary libraries
Mounted at /content/drive
Drive montato e percorso aggiunto con successo!


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = gym.make('LunarLander-v3')
state_shape = env.observation_space.shape
state_size = env.observation_space.shape[0]
number_actions = env.action_space.n
print('State shape: ', state_shape)
print('State size: ', state_size)
print('Number of actions: ', number_actions)

State shape:  (8,)
State size:  8
Number of actions:  4


In [8]:
number_episodes = 2000
n_samples_states = 50
eval_freq = 1
num_runs = 5

In [9]:
fixed_states = np.array([env.observation_space.sample() for _ in range(n_samples_states)])

In [10]:
configs = {

"baseline_soft": {
        "alpha": 1e-3,
        "update_type": "soft",
        "tau": 1e-3,
        "update_every": None,
        "gamma": 0.99,
        "epsilon_decay": 0.995
    },

    "low_lr_soft": {
        "alpha": 5e-4,
        "update_type": "soft",
        "tau": 1e-3,
        "update_every": None,
        "gamma": 0.99,
        "epsilon_decay": 0.995
    },
    "high_tau_soft": {
        "alpha": 1e-3,
        "update_type": "soft",
        "tau": 5e-3,
        "update_every": None,
        "gamma": 0.99,
        "epsilon_decay": 0.995
    },

    "baseline_hard": {
        "alpha": 1e-3,
        "update_type": "hard",
        "tau": None,
        "update_every": 10,
        "gamma": 0.99,
        "epsilon_decay": 0.999 # Increased epsilon_decay for slower decay
    },

    "low_lr_hard": {
        "alpha": 5e-4,
        "update_type": "hard",
        "tau": None,
        "update_every": 10,
        "gamma": 0.99,
        "epsilon_decay": 0.999 # Increased epsilon_decay for slower decay
    }
}

In [11]:
all_experiment_results = {exp_name: [] for exp_name in configs.keys()} #Dictionary which will contain all the results coming from the different training experiments

#For loop used to train multiple agents on the different configurations
for experiment_name, cfg in configs.items():
    for run_idx in range(num_runs):
        #Here the agent is instantiated using as parameters the ones already inside it and the others from the experiment's configurations
        agent = Agent(state_size,
                      number_actions,
                      alpha = cfg["alpha"],
                      target_update = cfg["update_type"],
                      tau = cfg["tau"],
                      target_update_steps = cfg["update_every"],
                      gamma = cfg["gamma"],
                      save_dir = save_path,
                      device = device)

        # Run training for the current experiment using the train_agent function
        experiment_results = train_agent(agent, experiment_name, cfg,
                                         number_episodes = number_episodes,
                                         run_idx = run_idx,
                                         env = env,
                                         fixed_states = fixed_states,
                                         num_runs = num_runs,
                                         eval_freq = eval_freq)

        #Append the results in the dictionary
        all_experiment_results[experiment_name].append(experiment_results)

env.close() #Close the environment

print("The training is now complete")


--- Starting training for experiment: baseline_soft (Run 1/5) ---
Episode 100	Average Score: -160.61

[SALVATAGGIO OK] Checkpoint saved for Run 1 at episode 100 in -> /content/drive/MyDrive/DQN project V3 - Copia/checkpoints
Episode 200	Average Score: -102.93

[SALVATAGGIO OK] Checkpoint saved for Run 1 at episode 200 in -> /content/drive/MyDrive/DQN project V3 - Copia/checkpoints
Episode 300	Average Score: -66.09

[SALVATAGGIO OK] Checkpoint saved for Run 1 at episode 300 in -> /content/drive/MyDrive/DQN project V3 - Copia/checkpoints
Episode 400	Average Score: -27.88

[SALVATAGGIO OK] Checkpoint saved for Run 1 at episode 400 in -> /content/drive/MyDrive/DQN project V3 - Copia/checkpoints
Episode 500	Average Score: 107.21

[SALVATAGGIO OK] Checkpoint saved for Run 1 at episode 500 in -> /content/drive/MyDrive/DQN project V3 - Copia/checkpoints
Episode 600	Average Score: 168.42

[SALVATAGGIO OK] Checkpoint saved for Run 1 at episode 600 in -> /content/drive/MyDrive/DQN project V3 - C